# Phase 5 — Model Training, Validation & Threshold Optimization
## Amazon ML Challenge 2026 — Business Entity Resolution

This notebook trains, evaluates, and tunes machine learning matching models on the 87-dimensional `phase4-v1` candidate pair feature space. The primary optimization metric is the challenge-official **Macro F0.5 at the Source 1 entity level**.

In [14]:
# 1. Environment & Module Setup
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import pickle
import numpy as np
import pandas as pd
from src.matching_features.feature_schema import FEATURE_COLUMNS, FEATURE_SCHEMA_VERSION
from src.matching_model.trainer import train_model, predict_probabilities
from src.matching_model.evaluation import evaluate_model_at_threshold, sweep_thresholds
from src.matching_model.feature_ablations import FEATURE_ABLATION_CONFIGS
from src.matching_model.error_analysis import analyze_model_errors
from src.matching_model.candidate_recall_analysis import compute_recall_ceilings

print(f'Feature Schema Version: {FEATURE_SCHEMA_VERSION} ({len(FEATURE_COLUMNS)} features)')

Feature Schema Version: phase4-v1 (87 features)


In [15]:
# 2. Data Loading
with open('../data_cache/train_features.pkl', 'rb') as f:
    train_df = pickle.load(f)
with open('../data_cache/val_features.pkl', 'rb') as f:
    val_df = pickle.load(f)
with open('../data_cache/ground_truth.pkl', 'rb') as f:
    train_gt, val_gt = pickle.load(f)

print(f'Loaded Train Pairs: {len(train_df):,}')
print(f'Loaded Val Pairs:   {len(val_df):,}')

Loaded Train Pairs: 155,024
Loaded Val Pairs:   156,200


In [16]:
# 3. Dataset Statistics & Label Distribution
train_pos = (train_df['label'] == 1).sum()
val_pos = (val_df['label'] == 1).sum()

print(f'Train Label Distribution: {train_pos:,} Positives ({train_pos/len(train_df)*100:.2f}%), {len(train_df)-train_pos:,} Negatives')
print(f'Val Label Distribution:   {val_pos:,} Positives ({val_pos/len(val_df)*100:.2f}%), {len(val_df)-val_pos:,} Negatives')

Train Label Distribution: 15,431 Positives (9.95%), 139,593 Negatives
Val Label Distribution:   15,372 Positives (9.84%), 140,828 Negatives


In [17]:
# 4. Feature Preparation & Integrity Verification
assert len(FEATURE_COLUMNS) == 87
assert train_df[FEATURE_COLUMNS].isna().sum().sum() == 0
assert val_df[FEATURE_COLUMNS].isna().sum().sum() == 0
print('All 87 features verified non-null and numeric.')

All 87 features verified non-null and numeric.


In [18]:
# 5. Model Training (Model Families)
models = {}
models['MODEL-001'] = train_model('MODEL-001', train_df)[0]
models['MODEL-002'] = train_model('MODEL-002', train_df)[0]
models['MODEL-003'] = train_model('MODEL-003', train_df)[0]
models['MODEL-004'] = train_model('MODEL-004', train_df)[0]
print('Trained Logistic Regression, Random Forest, HistGradientBoosting, and ExtraTrees.')

[2026-09-25 22:59:56] [INFO] [model_trainer]: Trained MODEL-001 in 1.96s on 155,024 samples (87 features).
[2026-09-25 23:00:01] [INFO] [model_trainer]: Trained MODEL-002 in 4.52s on 155,024 samples (87 features).
[2026-09-25 23:00:04] [INFO] [model_trainer]: Trained MODEL-003 in 2.89s on 155,024 samples (87 features).
[2026-09-25 23:00:07] [INFO] [model_trainer]: Trained MODEL-004 in 3.53s on 155,024 samples (87 features).
Trained Logistic Regression, Random Forest, HistGradientBoosting, and ExtraTrees.


In [19]:
# 6. Validation Predictions & Probability Estimation
probs = {}
for m_id, model in models.items():
    probs[m_id], inf_time = predict_probabilities(model, val_df)
    print(f'{m_id} inference complete in {inf_time}s.')

MODEL-001 inference complete in 0.09s.
MODEL-002 inference complete in 0.22s.
MODEL-003 inference complete in 0.2s.
MODEL-004 inference complete in 0.21s.


In [20]:
# 7. Threshold Sweep & Macro F0.5 Optimization
thresholds = [0.05 * i for i in range(1, 20)]
model_results = {}
for m_id in models:
    best_res, all_res = sweep_thresholds(val_df, probs[m_id], val_gt, thresholds=thresholds)
    model_results[m_id] = best_res
    print(f"{m_id} Optimal Threshold: {best_res['threshold']} -> Macro F0.5: {best_res['macro_f05']:.4f}")

MODEL-001 Optimal Threshold: 0.85 -> Macro F0.5: 0.9328
MODEL-002 Optimal Threshold: 0.6 -> Macro F0.5: 0.9310
MODEL-003 Optimal Threshold: 0.7 -> Macro F0.5: 0.9333
MODEL-004 Optimal Threshold: 0.55 -> Macro F0.5: 0.9310


In [21]:
# 8. Entity-Level Macro F0.5 Breakdown
best_res = model_results['MODEL-003']
print(f"Macro F0.5:           {best_res['macro_f05']:.4f}")
print(f"Macro Precision:      {best_res['macro_precision']:.4f}")
print(f"Macro Recall:         {best_res['macro_recall']:.4f}")
print(f"Singleton F0.5:       {best_res['singleton_f05']:.4f}")
print(f"Multi-match F0.5:     {best_res['multi_match_f05']:.4f}")
print(f"Zero-match Accuracy:  {best_res['zero_match_accuracy']:.4f}")

Macro F0.5:           0.9333
Macro Precision:      0.9646
Macro Recall:         0.8757
Singleton F0.5:       0.8709
Multi-match F0.5:     0.9356
Zero-match Accuracy:  0.9617


In [22]:
# 9. Model Comparison Summary Table
summary_df = pd.DataFrame(model_results.values())
summary_df = summary_df.sort_values(by='macro_f05', ascending=False)
summary_df[['threshold', 'macro_f05', 'macro_precision', 'macro_recall', 'roc_auc', 'pr_auc']]

,threshold,macro_f05,macro_precision,macro_recall,roc_auc,pr_auc
2,0.70,0.9333,0.9646,0.8757,0.9998,0.9985
0,0.85,0.9328,0.9660,0.8701,0.9998,0.9981
1,0.60,0.9310,0.9639,0.8700,0.9998,0.9978
3,0.55,0.9310,0.9646,0.8684,0.9998,0.9979


In [23]:
# 10. Feature Group Ablations
ablation_results = []
for exp_id, cfg in FEATURE_ABLATION_CONFIGS.items():
    m, _ = train_model('MODEL-003', train_df, feature_cols=cfg['columns'])
    p, _ = predict_probabilities(m, val_df, feature_cols=cfg['columns'])
    best_t, _ = sweep_thresholds(val_df, p, val_gt, thresholds=thresholds)
    ablation_results.append({
        'Experiment': exp_id,
        'Name': cfg['name'],
        'Features': cfg['count'],
        'Threshold': best_t['threshold'],
        'Macro F0.5': best_t['macro_f05'],
        'PR-AUC': best_t['pr_auc']
    })
pd.DataFrame(ablation_results)

[2026-09-25 23:00:23] [INFO] [model_trainer]: Trained MODEL-003 in 1.18s on 155,024 samples (15 features).
[2026-09-25 23:00:27] [INFO] [model_trainer]: Trained MODEL-003 in 1.3s on 155,024 samples (30 features).
[2026-09-25 23:00:32] [INFO] [model_trainer]: Trained MODEL-003 in 1.51s on 155,024 samples (38 features).
[2026-09-25 23:00:37] [INFO] [model_trainer]: Trained MODEL-003 in 1.69s on 155,024 samples (47 features).
[2026-09-25 23:00:42] [INFO] [model_trainer]: Trained MODEL-003 in 1.93s on 155,024 samples (58 features).
[2026-09-25 23:00:48] [INFO] [model_trainer]: Trained MODEL-003 in 2.0s on 155,024 samples (65 features).
[2026-09-25 23:00:54] [INFO] [model_trainer]: Trained MODEL-003 in 2.8s on 155,024 samples (87 features).
[2026-09-25 23:01:00] [INFO] [model_trainer]: Trained MODEL-003 in 2.48s on 155,024 samples (81 features).


,Experiment,Name,Features,Threshold,Macro F0.5,PR-AUC
0,EXP-015,Name Features Only,15,0.35,0.7873,0.7967
1,EXP-016,Name + Address,30,0.65,0.9324,0.9981
2,EXP-017,Name + Address + Cross-Field,38,0.80,0.9323,0.9980
3,EXP-018,All Similarity Features (Name+Addr+Cross+Count...,47,0.65,0.9326,0.9981
4,EXP-019,All Similarity + Provenance,58,0.65,0.9325,0.9982
5,EXP-020,All Similarity + Provenance + Refinement,65,0.65,0.9324,0.9982
6,EXP-021,All 87 Features (Full Feature Set),87,0.70,0.9333,0.9985
7,EXP-022,All Features Minus Frequency Features,81,0.60,0.9329,0.9986


In [24]:
# 11. Forensic Error Analysis on Best Model
errors = analyze_model_errors(val_df, probs['MODEL-003'], val_gt, threshold=best_res['threshold'])
print('False Positive Breakdown:', errors['false_positive_categories'])
print('False Negative Breakdown:', errors['false_negative_categories'])

False Positive Breakdown: {'same_address_different_business': 1, 'shared_commercial_complex': 11, 'generic_name_collision': 83, 'dba_confusion': 4, 'address_only_evidence': 1, 'legal_suffix_collisions': 93, 'other_false_positives': 8}
False Negative Breakdown: {'cross_script_names': 19, 'missing_address': 157, 'weak_address': 33, 'brand_renames': 0, 'severe_typos': 57, 'borderline_refinement_score': 91, 'other_false_negatives': 0}


In [25]:
# 12. Candidate Recall vs Model Discrimination Ceiling
ceilings = compute_recall_ceilings(val_df, probs['MODEL-003'], val_gt, threshold=best_res['threshold'])
for k, v in ceilings.items():
    print(f'{k}: {v}')

total_ground_truth_links: 17335
candidate_represented_true_links: 15372
model_captured_true_links: 15112
candidate_generation_recall_ceiling_pct: 88.68
model_recall_within_candidates_pct: 98.31
final_end_to_end_recall_pct: 87.18
blocking_miss_count: 1963
model_discrimination_miss_count: 260


In [ ]:
# 13. Reproducibility & Model Selection Summary
print('Selected Champion Model: MODEL-003 (HistGradientBoostingClassifier)')
print('Selected Threshold: 0.70')
print('Final Macro F0.5: 0.9333')
print('Phase 5 Complete. Ready for Phase 6 Pipeline Assembly.')

Selected Champion Model: MODEL-003 (HistGradientBoostingClassifier)
Selected Threshold: 0.70
Final Macro F0.5: 0.9333
Phase 5 Complete. Ready for Phase 6 Pipeline Assembly.


: 